## Ensemble_Agent Experimenting

In [4]:
from deal_hunter.agents.items import Item

In [5]:

#sys
from pathlib import Path
import sys
root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0,str(root_dir/"src"))


In [6]:
#imports
import os 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm



### Load Dataset

In [7]:
#hf login
load_dotenv(override = True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token , add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
pricer_data = "Vishy08/pricer-data"
train , test, val = Item.from_hub(dataset_name=pricer_data)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 320,000 training items, 4,000 validation items, 8,000 test items


In [9]:
test[0]

Item(title='ZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor (KPTT-30)', price=899.95, category='Appliances', test_prompt="How much does this cost to the nearest dollar?\n\nZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor\nThe ZLINE KPTT wall mount wooden range hood designed to be both elegant and powerful, featuring the industry's only lifetime warranty motor. The hand-finished painted cottage white wood is made from solid pine with a stainless steel inner frame, making it both durable and long-lasting. This durable construction, along with ZLINE's lifetime warranty on the motor, guarantees a range hood that will last for a lifetime. This classic wooden hood contains many modern features, such as Hand-carved, hand-finished wood; Dishwasher-safe stainless steel baffle filters; Built-in LED lighting; High performance motor with speeds up to 400 CFM. All ZLINE range hoods come equipped with everything needed to easily install and use,

### Testing ChromaDB

In [10]:
import chromadb
DB = "products_vectorstore"
chroma_client = chromadb.PersistentClient(path = DB)

In [11]:
import chromadb
chroma_client = chromadb.Client()

# switch \`create_collection\` to \`get_or_create_collection\` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(name="my_collection")

# switch \`add\` to \`upsert\` to avoid adding the same documents every time
collection.upsert(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1", "id2"]
)

results = collection.query(
    query_texts=["This is a query document about florida"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

print(results)

{'ids': [['id2', 'id1']], 'embeddings': None, 'documents': [['This is a document about oranges', 'This is a document about pineapple']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[1.1462141275405884, 1.3015384674072266]]}


### Encoder LLM (all-Mini-L6-v2)

In [12]:
#Using SentenceTransformer Encoding LLM ( Free(fast) + Private)
from sentence_transformers import SentenceTransformer
encoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7545.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
sentences = [
    item.test_prompt for item in test[:50]
]
embeddings = encoder_model.encode(sentences=sentences)


In [14]:
type(embeddings)

numpy.ndarray

### Building ChromaDB 

In [15]:
def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question,"").replace(prefix,"")
    return f"{summary}"

In [16]:
summarizer(test[42])

"Moving Out for Xbox One - Xbox One\nBecome a certified removals master in this action, puzzle, physics based moving Simulator that brings new meaning to couch co-op. In moving out players can train alone or with friends to learn the dos and don'ts of moving furniture. Expect plot twists, irreverent humor, cameos and borderline trademark infringing scenarios. You thought moving was dull, think again! Grab your friends and become a certified moving master in moving out! Moving out is currently in development as the premiere title from smog's melbourne studio and is made in collaboration with stockholm-based developer deem games. Co-op! Enjoy the game solo as an independent contractor or team up with some friends for the full story mode. Up to four players can cozy up on the couch together and argue over the best way to move.. A couch! Action! Players will learn the ropes in a"

In [ ]:
def build_product_collection(chroma_client , encoder_model, items, name="products", batch_size=500):

    collection = chroma_client.get_or_create_collection(name) 
    
    # processing data in batch of 1000 
    with tqdm(total = len(items) , desc=f"Chroma {name}:", unit = "doc",dynamic_ncols = True) as pbar:
        for start in range(0,len(items),batch_size):
            #creating the batch
            batch = items[start : start + batch_size] 
            
            #include title + description 
            docs = [summarizer(item) for item in batch]
            embeddings = encoder_model.encode(docs).astype(float).tolist()

            metas = [{"category" : item.category , "price" : item.price} for item in batch]
            doc_ids = [f"{name}_{start + k}" for k in range(len(batch))]

            collection.add(
                documents = docs,
                metadatas =metas,
                ids = doc_ids,
                embeddings = embeddings
            )
            pbar.update(len(batch))
    
    return collection


### Running and Similarity Search